# Train SAE and Interpret Themes

This notebook runs the full **In Your Own Words** pipeline on any text dataset:

1. **Load** — read texts from a CSV file  
2. **Embed** — encode texts with an OpenAI embedding model  
3. **Train SAE** — fit a sparse autoencoder on the embeddings  
4. **Interpret** — use an LLM to label each SAE dimension with a theme  
5. **Save** — write fidelity scores and theme labels to CSV  
6. **Annotate** — classify each response against the high-fidelity themes  
7. **Analyze** — visualize theme distributions across demographic groups  

**Requirements:** Set `OPENAI_API_KEY` in a `.env` file (or as an environment variable).

## Step 0 — Configuration

Edit these variables to point to your dataset and describe your survey question.

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
DATA_FILE   = "your_data.csv"   # path to your CSV
TEXT_COLUMN = "response"        # column containing free-text responses

# ── Survey context (used to generate LLM instructions) ────────────────────────
SURVEY_QUESTION = (
    "In at least 2-3 sentences, how would you describe your identity?"
)

# ── Embedding ─────────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-large"  # OpenAI embedding model
EMBEDDING_FILE  = "embeddings.npy"          # cache embeddings here to avoid re-computing

# ── SAE hyperparameters ───────────────────────────────────────────────────────
M = 32   # number of SAE dimensions (themes)
K = 4    # max active dimensions per response

# ── Output ────────────────────────────────────────────────────────────────────
CACHE_NAME      = "my_dataset"                              # name for LLM cache
CHECKPOINT_DIR  = f"sae_cache/checkpoints/{CACHE_NAME}"    # where to save SAE weights
OUTPUT_FILE     = f"sae_cache/{CACHE_NAME}_themes.csv"     # where to save final themes

## Step 1 — Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_KEY_SAE'] = os.getenv('OPENAI_API_KEY', '')

from hypothesaes.quickstart import train_sae
from hypothesaes.interpret_neurons import (
    NeuronInterpreter, InterpretConfig, LLMConfig, SamplingConfig
)

Path("sae_cache").mkdir(exist_ok=True)
print("Setup complete.")

## Step 2 — Load data

In [ ]:
df = pd.read_csv(DATA_FILE)

# Drop rows with missing text
df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)
texts = df[TEXT_COLUMN].tolist()

print(f"Loaded {len(texts)} responses from '{DATA_FILE}'")
print("\nSample responses:")
for t in texts[:3]:
    print(f"  • {t[:120]}")

## Step 3 — Embed texts

Embeddings are cached to `EMBEDDING_FILE` so re-running this cell is fast.

In [ ]:
from openai import OpenAI

def embed_texts(texts, model=EMBEDDING_MODEL, batch_size=100):
    """Embed a list of texts using the OpenAI API, in batches."""
    client = OpenAI()
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(input=batch, model=model)
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} texts", end="\r")
    print()
    return np.array(all_embeddings, dtype=np.float32)

if Path(EMBEDDING_FILE).exists():
    print(f"Loading cached embeddings from '{EMBEDDING_FILE}'")
    embeddings = np.load(EMBEDDING_FILE)
else:
    print(f"Embedding {len(texts)} texts with '{EMBEDDING_MODEL}'...")
    embeddings = embed_texts(texts)
    np.save(EMBEDDING_FILE, embeddings)
    print(f"Saved embeddings to '{EMBEDDING_FILE}'")

print(f"Embeddings shape: {embeddings.shape}")

## Step 4 — Train SAE and generate theme interpretations

In [ ]:
# Build generic LLM instructions from the survey question
TASK_INSTRUCTIONS = f"""All of the texts are responses to the question:
{SURVEY_QUESTION}
Features should describe a specific aspect of the response. For example:
- "mentions ..."
- "self-describes as ..."
- "discusses ..."
"""

# Train (or load from checkpoint if it already exists)
sae = train_sae(
    embeddings=embeddings,
    M=M,
    K=K,
    checkpoint_dir=CHECKPOINT_DIR,
)
print(f"SAE trained: {M} dimensions, {K} active per response")

In [ ]:
import sys
sys.path.insert(0, str(Path("../scripts").resolve()))
from train_sae import generate_interpretations_and_fidelity_scores

results_df = generate_interpretations_and_fidelity_scores(
    M=M,
    texts=texts,
    embeddings=embeddings,
    sae=sae,
    cache_name=CACHE_NAME,
    n_candidate_interpretations=3,
    task_specific_instructions=TASK_INSTRUCTIONS,
)

print("\nSample interpretations:")
for _, row in results_df.head(5).iterrows():
    print(f"  {row['interpretation']}  (F1={row['f1_fidelity_score']:.2f})")

In [ ]:
results_df

## Step 5 — Save theme results

In [ ]:
results_df = results_df.sort_values("f1_fidelity_score", ascending=False)
results_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(results_df)} theme interpretations to '{OUTPUT_FILE}'")
results_df

## Step 6 -- Annotate remaining themes

In [ ]:
from hypothesaes.annotate import annotate_texts_with_concepts

# Filter to themes with sufficient fidelity
FIDELITY_THRESHOLD = 0.5
themes = results_df[results_df["f1_fidelity_score"] >= FIDELITY_THRESHOLD]["interpretation"].tolist()
print(f"Annotating {len(themes)} themes with fidelity >= {FIDELITY_THRESHOLD}")

# Classify each response as activating each theme (1) or not (0)
annotations = annotate_texts_with_concepts(
    texts=texts,
    concepts=themes,
    max_words_per_example=128,
    cache_name=CACHE_NAME,
    n_workers=10,
    model="gpt-4o-mini",
)

annotations_df = pd.DataFrame(annotations)
ANNOTATIONS_FILE = f"sae_cache/{CACHE_NAME}_annotations.csv"
annotations_df.to_csv(ANNOTATIONS_FILE, index=False)

print(f"Saved {len(texts)} × {len(themes)} annotation matrix to '{ANNOTATIONS_FILE}'")
annotations_df.head()

## Step 7 — Analyze themes by categorical var

Use the stacked bar chart below to see which themes are more or less common across any categorical variable in your dataset (e.g. gender, race, age group, region).

Set `CATEGORY_COLUMN` to the name of the column you want to group by.

In [ ]:
from quickstart_figures import calculate_proportions, themes_barchart

# Set the categorical column you want to analyze
CATEGORY_COLUMN = "group"  # replace with a column in your df, e.g. 'gender', 'region'

# Compute the proportion of each theme activated by each group
proportions = calculate_proportions(annotations_df, df, CATEGORY_COLUMN)

# Plot a stacked bar chart — themes sorted by how group-specific they are
fig = themes_barchart(
    proportions,
    category_column=CATEGORY_COLUMN,
    sort_by="min_max_any_group",  # 'majority' or 'r2' also supported
)

### Multiple categories

To compare themes across several demographic variables at once, loop over a list of columns.

In [ ]:
CATEGORY_COLUMNS = ["group"]  # add more columns as needed, e.g. ["gender", "race", "region"]

for col in CATEGORY_COLUMNS:
    print(f"\n── {col} ──")
    proportions = calculate_proportions(annotations_df, df, col)
    themes_barchart(proportions, category_column=col, sort_by="min_max_any_group")